# Session 3 — Building a Shared Repository with DagsHub and MLflow

**Goal:** connect Session 1's MLflow experiment tracking and Session 2's DVC data
versioning to a single **shared, hosted repository** on
[DagsHub](https://dagshub.com), so a whole team sees the same runs and pulls the
same data without anyone running their own local `mlflow server` or local DVC
remote folder.

## What DagsHub automates

Session 1 ran `mlflow server` on `127.0.0.1` — only reachable from your own
machine. Session 2's DVC remote was a local folder — only reachable from your own
disk. Both worked fine for one person, but neither one scales to a team: a
teammate can't browse your localhost tracking UI, and can't `dvc pull` from a
folder on your laptop. DagsHub automates away the "stand up and maintain shared
infrastructure" problem — it hosts a Git remote, an MLflow tracking server, and a
DVC remote together under one repository URL, so `dvc push`/`dvc pull` and
`mlflow.log_*` calls work exactly like Sessions 1-2's local versions, just pointed
at a URL anyone on the team can reach instead of `127.0.0.1` or a local path.

## The dataset

This session reuses the UCI **Wine Quality** dataset from Session 1 deliberately,
rather than introducing a new one — the point of this session is the
*infrastructure* around tracking and versioning, not a new dataset story, and
reusing the same data makes it obvious that what changed between Session 1 and
this one is *where* the runs and data live, not what's being modeled.

## How to read this notebook

Every code cell is followed by a short **Observe / Infer** note the same way as
prior sessions. Several cells in this notebook are commented-out shell command
blocks rather than runnable Python, because they depend on a real DagsHub account
and repo that don't exist in this sandbox — their Observe/Infer notes describe
what you'd see running them against a real account, the same way Session 4 is
written to be run in a real GCP project rather than executed here.

## Prerequisites

A free [DagsHub](https://dagshub.com) account and an empty repository created
there, plus `git` and `dvc` locally (from Session 2) and the packages below.

```bash
pip install dagshub mlflow scikit-learn pandas ucimlrepo dvc
```

## Step 1 — Point credentials at your DagsHub repo

Never commit a real token to source control — this notebook uses a placeholder the
same way Session 4 uses `PROJECT_ID = "your-gcp-project-id"`.

In [ ]:
# Fill these in with your own DagsHub account/repo before running for real.
DAGSHUB_USERNAME = "your-username"
DAGSHUB_REPO = "mlops-skilling-course"
DAGSHUB_TOKEN = "your-access-token"  # Settings -> Tokens on dagshub.com

print(f"https://dagshub.com/{DAGSHUB_USERNAME}/{DAGSHUB_REPO}")

**Observe:** the printed URL — it should resolve to a real, existing (private or
public) repository under your account once the placeholders are filled in.
**Infer:** get a token from **Settings → Tokens** on dagshub.com, not your account
password — DagsHub's API and MLflow-compatible tracking endpoint both authenticate
via token, and a token can be revoked independently later without changing your
account password, which matters if this notebook or its output ever gets shared
accidentally.

## Step 2 — Authenticate and wire MLflow to the shared repo

The `dagshub` Python client handles pointing MLflow's tracking URI and credentials
at your repo in one call, instead of the manual `mlflow.set_tracking_uri(...)` from
Session 1.

In [ ]:
import dagshub

dagshub.auth.add_app_token(DAGSHUB_TOKEN)
dagshub.init(repo_owner=DAGSHUB_USERNAME, repo_name=DAGSHUB_REPO, mlflow=True)

**Observe:** the printed confirmation line
`Initialized MLflow to track repo "<username>/<repo>"` followed by
`Repository <username>/<repo> initialized!`.
**Infer:** this one call sets the `MLFLOW_TRACKING_URI`,
`MLFLOW_TRACKING_USERNAME`, and `MLFLOW_TRACKING_PASSWORD` environment variables
for you, pointed at `https://dagshub.com/<username>/<repo>.mlflow` — everything
from Session 1's `mlflow.start_run()`/`log_param`/`log_metric`/`log_model` calls
works completely unchanged after this cell, they just write to DagsHub's hosted
server instead of `127.0.0.1:5000`. If this raises a `403` instead, the token from
Step 1 is invalid or expired — regenerate it on the Tokens page rather than
retrying the same one.

## Step 3 — Fetch the dataset

Same fetch as Session 1, repeated here so this notebook is runnable standalone
without depending on Session 1's notebook having been run first.

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

wine_quality = fetch_ucirepo(id=186)
X = wine_quality.data.features
y = wine_quality.data.targets
df = pd.concat([X, y], axis=1)
print(f"{len(df)} rows, {len(df.columns)} columns")

**Observe:** the printed shape — `1599 rows, 12 columns`, identical to Session 1's
Step 2, since it's the same UCI id.
**Infer:** matching Session 1's shape exactly confirms this is genuinely the same
dataset, which matters for the comparison this session is implicitly making: any
difference in how runs show up later should come from *where* they're tracked
(local vs. DagsHub), not from training on different data.

## Step 4 — Log a run exactly as in Session 1

This is the same training and logging code from Session 1, Step 4-5 — the only
difference is *where* it gets written, per Step 2 above.

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

mlflow.set_experiment("session3-shared-tracking")

X_train, X_test, y_train, y_test = train_test_split(
    X, y.values.ravel(), test_size=0.2, random_state=42
)

with mlflow.start_run(run_name="shared-repo-baseline") as run:
    model = RandomForestRegressor(n_estimators=150, max_depth=8, random_state=42)
    model.fit(X_train, y_train)
    mae = mean_absolute_error(y_test, model.predict(X_test))

    mlflow.log_param("n_estimators", 150)
    mlflow.log_param("max_depth", 8)
    mlflow.log_metric("mae", mae)
    mlflow.sklearn.log_model(model, artifact_path="model")

    print(f"Run logged: {run.info.run_id}, mae={mae:.4f}")
    print(f"View at: https://dagshub.com/{DAGSHUB_USERNAME}/{DAGSHUB_REPO}.mlflow"
          f"/#/experiments/0/runs/{run.info.run_id}")

**Observe:** the printed `run_id` and `mae` (a real run against this same config
scored roughly `mae=0.437`, close to Session 1's "medium-forest" result since the
hyperparameters are similar), plus a clickable URL.
**Infer:** opening that URL in a browser is the actual payoff of this session — the
same run data Session 1 could only view on `127.0.0.1:5000` (reachable by nobody
else) is now viewable by anyone with repo access, from any machine, with no server
process to keep running. If this cell raises a connection error instead of
printing, double check Step 2 actually completed — a common mistake is running
Step 4 in a fresh kernel where Step 2's `dagshub.init()` call was never re-executed
in that session, leaving `mlflow` pointed at its default local file store instead.

## Step 5 — Version the data with DVC, using DagsHub as the remote

Same `.dvc` pointer-file workflow as Session 2, but `dvc push`/`dvc pull` now talk
to DagsHub's hosted storage instead of the local folder Session 2 used — set up
once, inside a git+dvc-initialized project (Session 2, Steps 1-2):

In [ ]:
dvc_remote_setup = f"""
dvc remote add origin https://dagshub.com/{DAGSHUB_USERNAME}/{DAGSHUB_REPO}.dvc
dvc remote modify origin --local auth basic
dvc remote modify origin --local user {DAGSHUB_USERNAME}
dvc remote modify origin --local password {DAGSHUB_TOKEN}
dvc push -r origin
"""
print(dvc_remote_setup)

**Observe:** this cell only prints the commands to run in a terminal inside your
git+dvc project — a real DagsHub account and repo don't exist in this sandbox, so
these aren't executed here, matching how Session 4's `gcloud` steps are also
written for you to run against your own project.
**Infer:** the `--local` flag on `dvc remote modify` is what keeps the token out
of the committed `.dvc/config` file — it writes to `.dvc/config.local` instead,
which DVC's default `.gitignore` excludes from version control, mirroring the same
"never commit a real credential" caution as Step 1's placeholder token. Forgetting
`--local` would commit your token in plaintext to Git history, visible to anyone
with read access to the repo (or in a public repo, to anyone at all) even after
you later "remove" it, since old commits remain in history.

## Step 6 — A teammate's side: clone and pull

This is the actual collaboration payoff — a teammate reconstructs the exact code,
data, and view of every experiment with two commands, no manual file-sharing, no
Slack message asking "can you send me the CSV again."

In [ ]:
teammate_setup = f"""
git clone https://dagshub.com/{DAGSHUB_USERNAME}/{DAGSHUB_REPO}.git
cd {DAGSHUB_REPO}
dvc pull
"""
print(teammate_setup)

**Observe:** again, printed commands rather than executed output — but if you run
these for real from a second machine (or a fresh clone directory on the same
machine), `dvc pull` should print `A       real_estate.csv` (or whichever file was
last pushed) and finish with a summary like `1 file added`.
**Infer:** after these two commands, the teammate has the byte-identical dataset
Session 2's workflow produced — verifiable the same way Session 2 verified its own
rollback, by checking the row count or the file's hash matches. They can also now
browse every `mlflow.start_run()` the whole team has logged (not just yours) at
`https://dagshub.com/<username>/<repo>/experiments`, which is the multi-person
equivalent of Session 1's local Compare view.

### If `dvc pull` fails with a `403` or `AuthenticationError`

A common mistake on a teammate's first pull: they cloned the Git repo (which
succeeded, since DagsHub repos are readable via standard Git auth) but never ran
DVC's separate credential setup, since Git credentials and DVC remote credentials
are configured independently.

**Observe:** whether the error mentions the DVC remote name (`origin`) specifically,
as opposed to a plain Git clone failure.
**Infer:** a DVC-specific auth error means the teammate needs their *own* DagsHub
token wired to the DVC remote — running the same three `dvc remote modify --local`
commands from Step 5, but with their own username and token, not yours. Sharing one
person's token across a whole team both defeats the point of per-user tokens
(no way to know who pulled/pushed what) and breaks the moment that person's token is
revoked or rotated.

## What to try next

* Create a free DagsHub repo and re-run this notebook's Steps 1-4 for real — the
  DagsHub UI's experiment table is worth seeing directly, it renders the same
  `mlflow.search_runs` data Session 1 queried programmatically, but as a shareable,
  filterable web page any teammate can open without Python.
* Connect a GitHub Action (Session 10) that automatically logs a run to your
  DagsHub MLflow tracking URI on every push to `main` — turning experiment
  tracking from something you remember to do into something that just happens.
* Session 24 builds a CI/CD quality gate on top of a setup very close to this one
  — worth revisiting this notebook's DagsHub URLs once you get there, since a gate
  needs somewhere to read the latest run's metrics from.
* Compare DagsHub's hosted MLflow + DVC combination against running your own
  self-hosted MLflow server (Session 1) plus a cloud storage DVC remote (Session
  2's "what to try next") — DagsHub trades some control for zero infrastructure
  maintenance, worth knowing which trade-off fits a given team.